# Import Libraries

In [1]:
# Standard Library
from pathlib import Path
from datetime import date
import warnings
import logging

# Data Handling
import numpy as np
import pandas as pd

# Data Download
import yfinance as yf

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8")
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")

Libraries imported successfully.


## Configuration

This section defines:

- Download date range
- Data directories
- Asset lists
- API settings

In [17]:
# ============================
# Date Configuration
# ============================

START_DATE = "1990-01-01"
END_DATE = date.today().strftime("%Y-%m-%d")

# ============================
# Project Directories
# ============================

PROJECT_ROOT = Path("..")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

STOCK_DIR = RAW_DATA_DIR / "stocks"
INDEX_DIR = RAW_DATA_DIR / "indices"
ETF_DIR = RAW_DATA_DIR / "etfs"
COMMODITY_DIR = RAW_DATA_DIR / "commodities"
FOREX_DIR = RAW_DATA_DIR / "forex"
MACRO_DIR = RAW_DATA_DIR / "macro"
NEWS_DIR = RAW_DATA_DIR / "news"

directories = [
    STOCK_DIR,
    INDEX_DIR,
    ETF_DIR,
    COMMODITY_DIR,
    FOREX_DIR,
    MACRO_DIR,
    NEWS_DIR
]

for directory in directories:
    directory.mkdir(parents=True, exist_ok=True)

# ============================
# Random Seed
# ============================

RANDOM_STATE = 42

# ============================
# Asset Lists
# ============================

STOCKS = [
    "AAPL",
    "MSFT",
    "NVDA",
    "GOOG",
    "AMZN",
    "META",
    "TSLA",
    "AMD"
]

INDICES = [
    "^GSPC",
    "^IXIC",
    "^DJI",
    "^RUT",
    "^VIX"
]

ETFS = [
    "SPY",
    "QQQ",
    "XLK",
    "SOXX",
    "VGT"
]

COMMODITIES = [
    "GC=F",
    "SI=F",
    "CL=F",
    "NG=F"
]

FOREX = [
    "EURUSD=X",
    "GBPUSD=X",
    "JPY=X",
    "USDCNY=X"
]

print("Configuration Loaded Successfully")

Configuration Loaded Successfully


## Data Downloader

Downloads market data from Yahoo Finance and saves each dataset
into its corresponding directory.

# Download Function

In [18]:
def download_market_data(
    tickers: list,
    save_directory: Path,
    asset_type: str
):
    """
    Download market data from Yahoo Finance.

    Parameters
    ----------
    tickers : list
        List of ticker symbols.

    save_directory : Path
        Output directory.

    asset_type : str
        Asset category.
    """

    summary = []

    print(f"\nDownloading {asset_type.upper()}...\n")

    for ticker in tickers:

        try:

            data = yf.download(
                ticker,
                start=START_DATE,
                end=END_DATE,
                auto_adjust=False,
                progress=False
            )

            if data.empty:
                print(f"{ticker} -> No Data")
                continue

            data.reset_index(inplace=True)

            filename = ticker.replace("^", "").replace("=", "_")

            filepath = save_directory / f"{filename}.csv"

            data.to_csv(filepath, index=False)

            summary.append({
                "Ticker": ticker,
                "Rows": len(data),
                "Start": data["Date"].min(),
                "End": data["Date"].max()
            })

            print(f"Saved -> {filepath.name}")

        except Exception as e:

            print(f"{ticker} -> {e}")

    return pd.DataFrame(summary)

## Download Financial Market Data

In [19]:
stocks_summary = download_market_data(
    STOCKS,
    STOCK_DIR,
    "Stocks"
)

indices_summary = download_market_data(
    INDICES,
    INDEX_DIR,
    "Indices"
)

etf_summary = download_market_data(
    ETFS,
    ETF_DIR,
    "ETFs"
)

commodity_summary = download_market_data(
    COMMODITIES,
    COMMODITY_DIR,
    "Commodities"
)

forex_summary = download_market_data(
    FOREX,
    FOREX_DIR,
    "Forex"
)



Saved -> AAPL.csv
Saved -> MSFT.csv
Saved -> NVDA.csv
Saved -> GOOG.csv
Saved -> AMZN.csv
Saved -> META.csv
Saved -> TSLA.csv
Saved -> AMD.csv


Saved -> GSPC.csv
Saved -> IXIC.csv
Saved -> DJI.csv
Saved -> RUT.csv
Saved -> VIX.csv


Saved -> SPY.csv
Saved -> QQQ.csv
Saved -> XLK.csv
Saved -> SOXX.csv
Saved -> VGT.csv


Saved -> GC_F.csv
Saved -> SI_F.csv
Saved -> CL_F.csv
Saved -> NG_F.csv


Saved -> EURUSD_X.csv
Saved -> GBPUSD_X.csv
Saved -> JPY_X.csv
Saved -> USDCNY_X.csv


# ########################################################################################################

# --------------------------------------------------------------------------------------------------------

# Download Macroeconomic Data

Macroeconomic indicators will be collected from the FRED database.

## Import for Macro

In [9]:
from pandas_datareader import data as web

## Macro Configuration

In [10]:
MACRO_INDICATORS = {
    "FED_RATE": "FEDFUNDS",
    "CPI": "CPIAUCSL",
    "UNEMPLOYMENT": "UNRATE",
    "GDP": "GDP",
    "PPI": "PPIACO",
    "DOLLAR_INDEX": "DTWEXBGS"
}

## Download Macro Data

In [11]:
macro_summary = []

for name, fred_code in MACRO_INDICATORS.items():

    try:

        df = web.DataReader(
            fred_code,
            "fred",
            START_DATE,
            END_DATE
        )

        filepath = MACRO_DIR / f"{name}.csv"

        df.to_csv(filepath)

        macro_summary.append({
            "Indicator": name,
            "Rows": len(df)
        })

        print(f"Saved -> {filepath.name}")

    except Exception as e:

        print(name, e)

macro_summary = pd.DataFrame(macro_summary)

Saved -> FED_RATE.csv
Saved -> CPI.csv
Saved -> UNEMPLOYMENT.csv
Saved -> GDP.csv
Saved -> PPI.csv
Saved -> DOLLAR_INDEX.csv


# -----------------------------------------------------------------------------------------------------

# Download Financial News

## News API Configuration 

In [14]:
NEWS_API_KEY = "YOUR_API_KEY"

## News Downloader

In [20]:

news_summary = []

for ticker in STOCKS:

    try:

        print(f"Downloading news for {ticker}...")

        news = yf.Ticker(ticker).news

        if not news:
            print(f"No news found for {ticker}")
            continue

        news_df = pd.DataFrame(news)

        news_df.to_csv(
            NEWS_DIR / f"{ticker}_news.csv",
            index=False
        )

        news_summary.append({
            "Ticker": ticker,
            "Articles": len(news_df)
        })

        print(f"Saved -> {ticker}_news.csv")

        time.sleep(1)

    except Exception as e:

        print(f"{ticker}: {e}")

news_summary = pd.DataFrame(news_summary)

Saved -> AAPL_news.csv
AAPL: name 'time' is not defined
Saved -> MSFT_news.csv
MSFT: name 'time' is not defined
Saved -> NVDA_news.csv
NVDA: name 'time' is not defined
Saved -> GOOG_news.csv
GOOG: name 'time' is not defined
Saved -> AMZN_news.csv
AMZN: name 'time' is not defined
Saved -> META_news.csv
META: name 'time' is not defined
Saved -> TSLA_news.csv
TSLA: name 'time' is not defined
Saved -> AMD_news.csv
AMD: name 'time' is not defined


# -----------------------------------------------------------------------------------------------------------

# Download Summary

## Summary

In [23]:
print("Stocks")
display(stocks_summary)

print("Indices")
display(indices_summary)

print("ETFs")
display(etf_summary)

print("Commodities")
display(commodity_summary)

print("Forex")
display(forex_summary)

print("Macro")
display(macro_summary)

print("News")
display(news_summary)

Stocks


,Ticker,Rows,Start,End
0,AAPL,9198,1990-01-02,2026-07-13
1,MSFT,9198,1990-01-02,2026-07-13
2,NVDA,6909,1999-01-22,2026-07-13
3,GOOG,5508,2004-08-19,2026-07-13
4,AMZN,7334,1997-05-15,2026-07-13
5,META,3556,2012-05-18,2026-07-13
6,TSLA,4033,2010-06-29,2026-07-13
7,AMD,9198,1990-01-02,2026-07-13


Indices


,Ticker,Rows,Start,End
0,^GSPC,9198,1990-01-02,2026-07-13
1,^IXIC,9198,1990-01-02,2026-07-13
2,^DJI,8692,1992-01-02,2026-07-13
3,^RUT,9198,1990-01-02,2026-07-13
4,^VIX,9199,1990-01-02,2026-07-13


ETFs


...

,Ticker,Rows,Start,End
0,EURUSD=X,5868,2003-12-01,2026-07-14
1,GBPUSD=X,5880,2003-12-01,2026-07-14
2,JPY=X,7702,1996-10-30,2026-07-14
3,USDCNY=X,6270,2001-06-25,2026-07-14


Macro


,Indicator,Rows
0,FED_RATE,438
1,CPI,437
2,UNEMPLOYMENT,438
3,GDP,145
4,PPI,437
5,DOLLAR_INDEX,5355


News


,Ticker,Articles
0,AAPL,10
1,MSFT,10
2,NVDA,10
3,GOOG,10
4,AMZN,10
5,META,10
6,TSLA,10
7,AMD,10
